In [3]:
import pandas as pd 
import numpy as np

df = pd.read_parquet("../data/dataset.parquet")
df['timestamp'] = pd.to_datetime(df['timestamp'])   

In [4]:
# Adding Calendar Features

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

In [5]:
df.head()

,timestamp,load,temperature,humidity,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,...,QV2M_dav,TQL_dav,W2M_dav,Holiday_ID,holiday,school,hour,day_of_week,month,is_weekend
0,2015-01-03 01:00:00,970.3450,25.865259,0.018576,0.016174,21.850546,23.482446,0.017272,0.001855,10.328949,...,0.016562,0.096100,5.364148,0,0,0,1,5,1,1
1,2015-01-03 02:00:00,912.1755,25.899255,0.018653,0.016418,22.166944,23.399255,0.017265,0.001327,10.681517,...,0.016509,0.087646,5.572471,0,0,0,2,5,1,1
2,2015-01-03 03:00:00,900.2688,25.937280,0.018768,0.015480,22.454911,23.343530,0.017211,0.001428,10.874924,...,0.016479,0.078735,5.871184,0,0,0,3,5,1,1
3,2015-01-03 04:00:00,889.9538,25.957544,0.018890,0.016273,22.110481,23.238794,0.017128,0.002599,10.518620,...,0.016487,0.068390,5.883621,0,0,0,4,5,1,1
4,2015-01-03 05:00:00,893.6865,25.973840,0.018981,0.017281,21.186089,23.075403,0.017059,0.001729,9.733589,...,0.016456,0.064362,5.611724,0,0,0,5,5,1,1


In [10]:
# Holiday Features
!pip3 install holidays
import holidays
panama_holidays = holidays.country_holidays("PA")

df["is_holiday"] = df["timestamp"].dt.date.astype("datetime64[ns]").isin(panama_holidays)

df["is_holiday"] = df["is_holiday"].astype(int)

In [11]:
# Cyclical Encoding 
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dayofweek_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dayofweek_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [12]:
df.head()

,timestamp,load,temperature,humidity,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,...,day_of_week,month,is_weekend,is_holiday,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,month_sin,month_cos
0,2015-01-03 01:00:00,970.3450,25.865259,0.018576,0.016174,21.850546,23.482446,0.017272,0.001855,10.328949,...,5,1,1,0,0.258819,0.965926,-0.974928,-0.222521,0.5,0.866025
1,2015-01-03 02:00:00,912.1755,25.899255,0.018653,0.016418,22.166944,23.399255,0.017265,0.001327,10.681517,...,5,1,1,0,0.500000,0.866025,-0.974928,-0.222521,0.5,0.866025
2,2015-01-03 03:00:00,900.2688,25.937280,0.018768,0.015480,22.454911,23.343530,0.017211,0.001428,10.874924,...,5,1,1,0,0.707107,0.707107,-0.974928,-0.222521,0.5,0.866025
3,2015-01-03 04:00:00,889.9538,25.957544,0.018890,0.016273,22.110481,23.238794,0.017128,0.002599,10.518620,...,5,1,1,0,0.866025,0.500000,-0.974928,-0.222521,0.5,0.866025
4,2015-01-03 05:00:00,893.6865,25.973840,0.018981,0.017281,21.186089,23.075403,0.017059,0.001729,9.733589,...,5,1,1,0,0.965926,0.258819,-0.974928,-0.222521,0.5,0.866025


In [13]:
# Lag Features
df["lag_1"] = df["load"].shift(1)
df["lag_24"] = df["load"].shift(24)
df["lag_168"] = df["load"].shift(168)

In [14]:
# Rolling Statistics 
df["rolling_mean_24"] = df["load"].rolling(window=24).mean()
df["rolling_std_24"] = df["load"].rolling(window=24).std()

df["rolling_mean_168"] = df["load"].rolling(window=168).mean()

In [15]:
df.head()

,timestamp,load,temperature,humidity,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,...,dayofweek_sin,dayofweek_cos,month_sin,month_cos,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168
0,2015-01-03 01:00:00,970.3450,25.865259,0.018576,0.016174,21.850546,23.482446,0.017272,0.001855,10.328949,...,-0.974928,-0.222521,0.5,0.866025,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-03 02:00:00,912.1755,25.899255,0.018653,0.016418,22.166944,23.399255,0.017265,0.001327,10.681517,...,-0.974928,-0.222521,0.5,0.866025,970.3450,NaN,NaN,NaN,NaN,NaN
2,2015-01-03 03:00:00,900.2688,25.937280,0.018768,0.015480,22.454911,23.343530,0.017211,0.001428,10.874924,...,-0.974928,-0.222521,0.5,0.866025,912.1755,NaN,NaN,NaN,NaN,NaN
3,2015-01-03 04:00:00,889.9538,25.957544,0.018890,0.016273,22.110481,23.238794,0.017128,0.002599,10.518620,...,-0.974928,-0.222521,0.5,0.866025,900.2688,NaN,NaN,NaN,NaN,NaN
4,2015-01-03 05:00:00,893.6865,25.973840,0.018981,0.017281,21.186089,23.075403,0.017059,0.001729,9.733589,...,-0.974928,-0.222521,0.5,0.866025,889.9538,NaN,NaN,NaN,NaN,NaN


In [16]:
# We need to drop the NaN values created by lag and rolling features
df = df.dropna().reset_index(drop=True)

In [17]:
df.head()

,timestamp,load,temperature,humidity,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,...,dayofweek_sin,dayofweek_cos,month_sin,month_cos,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168
0,2015-01-10 01:00:00,906.9580,24.976495,0.017215,0.025253,21.335333,23.554620,0.016421,0.051376,12.485627,...,-0.974928,-0.222521,0.5,0.866025,949.5031,943.6081,970.3450,997.486758,80.323661,1091.670407
1,2015-01-10 02:00:00,863.5135,24.906274,0.017337,0.034378,22.177057,23.429712,0.016337,0.038712,12.949576,...,-0.974928,-0.222521,0.5,0.866025,906.9580,917.0640,912.1755,995.255488,83.341887,1091.380752
2,2015-01-10 03:00:00,848.4447,24.879724,0.017512,0.045349,22.742188,23.309412,0.016292,0.028526,13.091533,...,-0.974928,-0.222521,0.5,0.866025,863.5135,895.9092,900.2688,993.277800,86.312089,1091.072275
3,2015-01-10 04:00:00,839.8821,24.907922,0.017673,0.047501,22.630025,23.306360,0.016337,0.016182,13.050650,...,-0.974928,-0.222521,0.5,0.866025,848.4447,885.2720,889.9538,991.386554,89.229555,1090.774229
4,2015-01-10 05:00:00,847.1073,24.932520,0.017795,0.036087,21.731658,23.354395,0.016422,0.009590,12.846088,...,-0.974928,-0.222521,0.5,0.866025,839.8821,884.8111,893.6865,989.815562,91.490782,1090.496972


In [18]:
df.shape

(47880, 34)

In [19]:
### Save the final dataset file 
df.to_parquet("../data/processed/panama_features.parquet", index=False)